# 03 — Explore the Data

Now that we've tested our functions, let's use them on a larger dataset.

This notebook shows how the same functions you unit-tested locally work on real-ish data at scale.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from baseball_stats import (
    add_batting_average,
    add_slugging_pct,
    filter_qualified_batters,
    top_n_by_stat,
    aggregate_team_stats,
    batting_average,
    slugging_percentage,
    on_base_percentage,
    ops,
    classify_hitter,
)

spark = SparkSession.builder.master("local[*]").appName("explore").getOrCreate()

In [ ]:
# A fuller dataset — 2025 Spring Training batting stats (fictional)
schema = StructType([
    StructField("player",    StringType(),  True),
    StructField("team",      StringType(),  True),
    StructField("hits",      IntegerType(), True),
    StructField("at_bats",   IntegerType(), True),
    StructField("singles",   IntegerType(), True),
    StructField("doubles",   IntegerType(), True),
    StructField("triples",   IntegerType(), True),
    StructField("home_runs", IntegerType(), True),
])

data = [
    # NL West
    ("Mookie Betts",       "Dodgers",   38, 110,  22, 8,  1,  7),
    ("Shohei Ohtani",      "Dodgers",   42, 105,  18, 10, 2, 12),
    ("Freddie Freeman",    "Dodgers",   35, 100,  22, 9,  0,  4),
    ("Corbin Carroll",     "D-backs",   30,  95,  20, 5,  2,  3),
    ("Ketel Marte",        "D-backs",   33,  98,  18, 8,  1,  6),
    ("Manny Machado",      "Padres",    28,  92,  16, 6,  0,  6),
    ("Fernando Tatis Jr",  "Padres",    31,  88,  15, 7,  1,  8),
    ("Heliot Ramos",       "Giants",    25,  85,  17, 4,  1,  3),
    ("Ezequiel Tovar",     "Rockies",   27,  90,  18, 5,  2,  2),
    # AL West
    ("Mike Trout",         "Angels",    22,  72,  12, 5,  0,  5),
    ("Julio Rodriguez",    "Mariners",  29,  95,  17, 6,  1,  5),
    ("Corey Seager",       "Rangers",   34, 102,  20, 7,  1,  6),
    ("Marcus Semien",      "Rangers",   30,  98,  19, 6,  2,  3),
    ("Yordan Alvarez",     "Astros",    36,  95,  16, 9,  0, 11),
    ("Kyle Tucker",        "Astros",    32,  90,  19, 7,  1,  5),
    # NL/AL East
    ("Aaron Judge",        "Yankees",   35, 100,  15, 6,  0, 14),
    ("Juan Soto",          "Yankees",   38, 102,  24, 8,  1,  5),
    ("Ronald Acuna Jr",    "Braves",    34,  98,  22, 6,  2,  4),
    ("Francisco Lindor",   "Mets",      30,  95,  18, 7,  1,  4),
    ("Bobby Witt Jr",      "Royals",    40, 108,  25, 8,  3,  4),
    # Light hitters for testing edge cases
    ("Backup Catcher",     "Rockies",    3,  25,   2, 1,  0,  0),
    ("September Callup",   "Giants",     1,  12,   1, 0,  0,  0),
    ("Pinch Runner",       "D-backs",    0,   5,   0, 0,  0,  0),
]

df = spark.createDataFrame(data, schema)
print(f"{df.count()} players loaded")
df.show(5)

## Apply the transformations we tested

In [ ]:
# Chain the transformations
stats = add_slugging_pct(add_batting_average(df))

print("All players with computed stats:")
stats.select("player", "team", "hits", "at_bats", "batting_avg", "slugging_pct").show(25, truncate=False)

In [ ]:
# Filter to qualified batters only
qualified = filter_qualified_batters(stats, min_at_bats=50)
print(f"Qualified batters (50+ AB): {qualified.count()} of {stats.count()}")

# Top 5 by batting average
print("\nBatting Average Leaders:")
top_n_by_stat(qualified, "batting_avg", n=5) \
    .select("player", "team", "batting_avg") \
    .show(truncate=False)

# Top 5 by slugging
print("Slugging Leaders:")
top_n_by_stat(qualified, "slugging_pct", n=5) \
    .select("player", "team", "slugging_pct") \
    .show(truncate=False)

In [ ]:
# Team aggregates
print("Team Stats:")
team_stats = aggregate_team_stats(stats)
team_stats.orderBy("total_home_runs", ascending=False).show(truncate=False)

## Use the pure Python functions too

In [ ]:
# Compute OPS for a few players using the pure Python functions
players = [
    {"name": "Shohei Ohtani", "H": 42, "AB": 105, "BB": 18, "HBP": 2, "SF": 3,
     "1B": 18, "2B": 10, "3B": 2, "HR": 12},
    {"name": "Aaron Judge",   "H": 35, "AB": 100, "BB": 22, "HBP": 1, "SF": 2,
     "1B": 15, "2B": 6,  "3B": 0, "HR": 14},
    {"name": "Bobby Witt Jr", "H": 40, "AB": 108, "BB": 10, "HBP": 1, "SF": 4,
     "1B": 25, "2B": 8,  "3B": 3, "HR": 4},
]

print(f"{'Player':<20} {'AVG':>6} {'OBP':>6} {'SLG':>6} {'OPS':>6}  {'Tier'}")
print("-" * 65)

for p in players:
    avg = batting_average(p["H"], p["AB"])
    obp = on_base_percentage(p["H"], p["BB"], p["HBP"], p["AB"], p["SF"])
    slg = slugging_percentage(p["1B"], p["2B"], p["3B"], p["HR"], p["AB"])
    o = ops(obp, slg)
    tier = classify_hitter(avg)
    print(f"{p['name']:<20} {avg:>6.3f} {obp:>6.3f} {slg:>6.3f} {o:>6.3f}  {tier}")

## The Point

Every function used in this notebook was **unit tested first** — both the pure Python math and the Spark transformations.

If someone changes `add_batting_average()` and breaks the formula, the test suite catches it before it ever runs on real data. That's the value of unit tests: **confidence that your code does what you think it does.**